# Main Quest 03 · 한글 자모 배치 생성

작성자: 강지수 · AIFFEL 리서처 과정 19기 · 2026-09-23

이 노트북은 **실제 수행한 실험을 설명하고 다시 실행하기 위한 제출본**입니다. 원실험의 원시 예측·학습 로그를 저장소에 보존했습니다. 아래의 결과 설명은 그 기록을 인용한 것이며, 실행하지 않은 셀에 출력을 만들어 넣지 않았습니다.

[실제 실행 Colab](https://colab.research.google.com/drive/18lUqbarOzuOhBdfQMb5ozs56AIAYQ8fb) · [논문 PDF](https://github.com/kritik-sowieso/AIFFEL_quest_rs/blob/main/MainQuest/Quest03/온19기_MainQuest3_강지수.pdf)

**결과: A/B/C 모두 테스트 배치 정답 0/25. 좌표 보조학습의 개선 효과는 확인하지 못했습니다.**

- [임시 데모](https://cfe69f138f13baaae8.gradio.live) · Colab 세션 종료 시 중단
- 실제 B 1회차 어댑터 사용 · 저장 출력 3/3 재현 확인
- 테스트 정답 0/25 · 원시 출력 관찰용 연구 시연


## 1. 연구 질문

LLM이 분리된 자모의 뜻을 읽는 것과 같은 방식으로 정확하게 생성하는 것은 다릅니다. 이번에는 **각 음절의 윗줄을 맞춘 한 가지 자모 배치 생성**만 평가합니다.

- A: 추가학습 전 Qwen2.5-0.5B-Instruct
- B: 정상 한글 → 자모 배치 + 자모 목록 보조학습
- C: 같은 변환 + 같은 자모 목록에 좌표를 포함하는 보조학습

주 비교는 C 대 B입니다. 좌표는 학습할 정답에만 포함됩니다. 추론 입력에는 정상 한글과 공통 지시만 들어갑니다. 수업 연결은 사전학습 모델(backbone)을 과업별 자료로 미세 조정(fine-tuning)하는 HuggingFace 커스텀 프로젝트 흐름입니다.

## 2. 전체 흐름

```text
[한글 공간 구조를 반영하는 생성 과제 정의]
                       |
                       v
[배치 규칙 고정 + 데이터/채점 코드 검사]
                       |
          +------------+-------------+
          |                          |
          v                          v
[GPU 1단계 파일럿 성공]    [EasyOCR 파일럿 10개]
          |                          |
          |                [원시 자모 보존 0/10]
          |                          |
          |                [한계·후속 연구로 기록]
          v
[합성 원문·자모 정답·좌표 248개]
                       |
          +------------+-------------+
          |            |             |
          v            v             v
    [학습 198개]  [검증 25개]   [테스트 25개]
          |            |             |
          v            |       선택 고정까지 보류
 [동일 사전학습 모델]   |             |
          |            |             |
     +----+----+       |             |
     |         |       |             |
     v         v       |             |
[B: 변환 +  [C: 변환 + |             |
 자모 목록]  자모·좌표] |             |
     |         |       |             |
 [각 5 epoch, 같은 학습 설정]        |
     |         |       |             |
     +----+----+-------+             |
          |                          |
          v                          |
[검증으로 B/C 체크포인트·데모 후보 선택]
          |                          |
          v                          v
[선택 기록 고정] --> [A/B/C 동일 테스트 평가]
                              |
                   +----------+----------+
                   |                     |
                   v                     v
           [결과표·오류 분석]   [선택된 실제 어댑터]
                   |                     |
                   v                     v
          [영문 논문·Overleaf]  [Colab 임시 데모]
                   |                     |
                   +----------+----------+
                              |
                              v
           [검수·공개 승인 -> GitHub 제출 자료]
                              |
                              v
           [후속: 상시 모델 호스팅과 웹 배포]
```

## 3. OCR 파일럿과 연구 질문의 변경

EasyOCR 한국어+영어 설정으로 합성 이미지 10개를 읽었습니다. 원시 자모 목록 완전 일치 **0/10**, 검출 없음 **2/10**이었습니다. 그래서 본 실험은 OCR이 아닌 **합성 정답의 좌표**를 사용합니다. OCR 성능 개선이라고 주장하지 않습니다.

T4에서 짧은 예시의 LoRA 한 단계 실행은 성공했습니다. 이 사실은 학습 코드의 실행 가능성을 뜻하며 생성 정확도를 뜻하지 않습니다.

## 4. 실제 결과와 모델 선택

| 조건 | 배치 정답 | 자모 목록 완전 보존 | 출력 길이 제한 | 평균 생성 시간 |
|---|---:|---:|---:|---:|
| A | 0/25 | 0/25 | 8/25 | 2.38초 |
| B | 0/25 | 0/25 | 4/25 | 1.65초 |
| C | 0/25 | 0/25 | 6/25 | 2.28초 |

B/C 각 5회차의 검증 정답 수와 자모 보존 수가 모두 0이었습니다. 사전에 정한 동점 규칙으로 **B/C 모두 1회차**, 데모 후보는 B가 선택됐습니다. 1회차가 더 우수하다는 뜻은 아닙니다. 테스트 결과로 선택 기준을 바꾸지 않았습니다.

학습 변환 손실은 B 3.745→1.346, C 3.581→1.317로 낮아졌습니다. 하지만 앞 정답 토큰을 보고 다음 토큰을 예측하는 학습 손실 감소가 스스로 연속 생성하는 성공으로 이어지지는 않았습니다.

![실제 학습 곡선과 출력 제한](https://raw.githubusercontent.com/kritik-sowieso/AIFFEL_quest_rs/main/MainQuest/Quest03/paper/figures/main_results.png)

## 5. 재현 실행 전 확인

기본값은 재학습·공개 데모를 시작하지 않습니다. 결과만 읽는다면 아래 셀을 실행할 필요가 없습니다. 재현하려면 Colab의 GPU 런타임을 선택하고 `RUN_TRAINING=True`로 바꾸세요. 지정 공개 저장소에서 이 연구의 코드·자료를 내려받고 라이브러리를 설치합니다. Drive 전체 연결이나 API 키는 필요하지 않습니다.

데모만 실행하려면 `RUN_DEMO=True`로 바꾸세요. 이 경우 Colab에서 임시 공개 링크를 생성합니다. 모델의 정답률은 0/25이며 원시 출력을 관찰하는 용도입니다. 세션 종료 시 링크가 중단됩니다.

In [ ]:
# 원하는 실행만 명시적으로 켭니다. 둘 다 False면 읽기용으로 사용합니다.
RUN_TRAINING = False
RUN_DEMO = False


## 6. 코드·자료 준비

이미 있는 실험 결과를 덮어쓰지 않도록 학습 작업 폴더를 별도로 만듭니다. 같은 폴더가 있으면 중단합니다. 공개 저장소 원본은 보존합니다.

In [ ]:
import subprocess
import sys
import shutil
from pathlib import Path

SOURCE_ROOT = Path('/content/mq03_source')
PROJECT = SOURCE_ROOT / 'MainQuest/Quest03'
RUN_ROOT = Path('/content/mq03_reproduction_run')

if RUN_TRAINING or RUN_DEMO:
    if not SOURCE_ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/kritik-sowieso/AIFFEL_quest_rs.git',
                        str(SOURCE_ROOT)], check=True)
    requirements = PROJECT / ('requirements-colab.txt' if RUN_DEMO else 'requirements-training.txt')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
    print('연구 코드와 라이브러리 준비 완료')
else:
    print('재학습·데모 실행을 요청하지 않아 준비 단계를 건너뜁니다.')


## 7. 자료 분할과 정답 규칙

학습 198 / 검증 25 / 테스트 25개입니다. 같은 단어를 그대로 포함하는 문구는 같은 분할로 묶었습니다. 음절·형태소 중복은 남으므로 새 자모나 새 스타일 일반화 실험은 아닙니다.

정답 자모 한 개를 한 셀에 두고 음절 사이 한 셀을 비웁니다. 세로 모음은 초성 오른쪽, 가로 모음과 받침은 아래로 배치합니다. 복합 방향 모음은 제외합니다. 정답 규칙은 데이터·채점에 사용하며 데모에서 모델 대신 정답을 생성하지 않습니다.

핵심 파일: `jamo/core.py`, `data/train_reviewed.jsonl`, `data/validation.jsonl`, `data/test.jsonl`.

In [ ]:
if RUN_TRAINING:
    if RUN_ROOT.exists():
        raise RuntimeError('이미 작업 폴더가 있습니다. 기존 결과를 보존한 채 새 경로를 지정하세요.')
    RUN_ROOT.mkdir()
    # 코드만 복사하고 기존 원시 평가 결과와 선택 파일은 가져오지 않습니다.
    for folder in ['jamo', 'training', 'evaluation', 'demo', 'tests']:
        for source in (PROJECT / folder).rglob('*.py'):
            target = RUN_ROOT / source.relative_to(PROJECT)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
    (RUN_ROOT / 'data').mkdir()
    for name in ['train_reviewed.jsonl', 'validation.jsonl', 'test.jsonl']:
        shutil.copy2(PROJECT / 'data' / name, RUN_ROOT / 'data' / name)
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                   cwd=RUN_ROOT, check=True)
    print('분할·정답·채점 검사 완료')


## 8. 학습에서 꼭 이해할 부분

LoRA는 기반 모델의 큰 가중치를 고정하고 작은 저랭크 행렬만 학습합니다. B/C는 같은 초기 모델·원문·주 과제 정답·학습 회차·난수 시드(seed)를 사용합니다.

`training/train.py`의 핵심 계산은 다음과 같습니다.

```python
# 입력 지시는 손실 계산에서 제외하고 출력 정답만 학습합니다.
labels[:, :len(prefix)] = -100
# 주 과제와 보조 과제의 토큰 평균 손실을 각각 0.5 가중합니다.
(loss * 0.5 / len(group)).backward()
```

원문 8개마다 기울기를 누적해 한 번 업데이트합니다. 전체 자료를 한 번 보는 회차를 epoch라고 하며, B/C 각 5 epoch와 125회 업데이트를 수행합니다. C의 보조 목표는 좌표 때문에 길어지므로 같은 횟수라도 처리 토큰·시간이 같지는 않습니다.

고정 설정: rank 8 / alpha 16 / dropout 0 / q·v 투영층 / 학습률 0.0002 / float32 / seed 20260923.

## 9. 학습 → 검증 선택 고정 → 테스트

아래 실행기는 B 5회 학습과 회차별 검증, C 5회 학습과 검증, A 검증을 순서대로 수행합니다. 엄격 배치 정답 수 → 자모 보존 수 → 더 이른 회차 순으로 선택한 기록을 저장한 뒤 테스트를 한 번씩 평가합니다. 동점이면 데모는 B를 선택합니다. 원시 예측을 덮어쓰지 않습니다.

원실험의 전체 소요 시간은 약 26분이었고 상한은 90분입니다. 다른 런타임의 시간·수치가 정확히 같음을 보장하지 않습니다.

In [ ]:
if RUN_TRAINING:
    # 이 셀은 실제 GPU 학습을 수행합니다. 기본값 False에서는 실행하지 않습니다.
    subprocess.run([sys.executable, '-u', '-m', 'training.reproduce'], cwd=RUN_ROOT, check=True)
    print('학습·고정 선택·테스트 평가 완료')


## 10. 정답률을 해석하는 방법

CRLF, 줄 끝 ASCII 공백, 바깥 빈 줄만 정규화합니다. 내부 공백, 설명 문장, 코드 펜스는 제거하지 않습니다. 자모를 다 써도 위치가 틀리면 배치 정답이 아닙니다. 자모 목록 보존은 별도의 지표입니다.

이번에는 자모 목록도 모두 틀렸으므로 공백 채점이 엄격해서만 실패했다고 볼 수 없습니다. 아래는 `달빛`의 실제 원시 출력 예시입니다.

![실패 예시](https://raw.githubusercontent.com/kritik-sowieso/AIFFEL_quest_rs/main/MainQuest/Quest03/paper/figures/error_example.png)

In [ ]:
import json

if RUN_TRAINING:
    summary = json.loads((RUN_ROOT / 'evaluation/summary.json').read_text())
    choice = json.loads((RUN_ROOT / 'evaluation/frozen_selection.json').read_text())
    print('검증으로 고정한 선택:', choice)
    print('이번 재현 실행의 실제 결과:', summary)
    # 원실험과 재현 실행을 혼동하지 않도록 새 결과는 별도 폴더에 남깁니다.
    from google.colab import files
    files.download(str(RUN_ROOT / 'main_experiment_results.zip'))


## 11. 선택된 원실험 모델의 임시 데모

- 모델: 공개 자료에 포함된 원실험 B 1회차 · 재현 학습에서 새로 선택한 모델과 구분
- 검수: 실제 가중치 지문 확인 · 저장 출력 3개와 일치
- 출력: 모델 원문 유지 · 정답 교정·대체 없음
- 입력 한도: 24자 · 실제 테스트 길이 2~8자
- 평가 결과: 테스트 배치 정답 0/25
- 기본 실행: `RUN_DEMO=False` · 자동 공개 없음
- 실행 방법: 임시 공개에 동의할 때 `RUN_DEMO=True`로 변경 후 실행
- 운영 조건: Colab 세션 유지 · 종료 시 임시 주소 접속 중단
- [현재 임시 데모](https://cfe69f138f13baaae8.gradio.live)
- [운영·검수 기록](runs/DEMO_RECORD.md)


In [ ]:
import os

if RUN_DEMO:
    demo_log = open('/content/mq03_demo.log', 'w')
    # 통계 수집을 끄고 실제 모델 출력만 전달합니다. 사용자 입력을 별도 저장하지 않습니다.
    demo_env = dict(os.environ, GRADIO_ANALYTICS_ENABLED='False', PYTHONUNBUFFERED='1')
    demo_process = subprocess.Popen([sys.executable, '-u', '-m', 'demo.app', '--share'],
                                    cwd=PROJECT, stdout=demo_log, stderr=subprocess.STDOUT, env=demo_env)
    print('아래 셀에서 임시 링크를 확인하세요.')


In [ ]:
if RUN_DEMO:
    print(Path('/content/mq03_demo.log').read_text()[-5000:])


In [ ]:
# 데모를 중지하려면 STOP_DEMO를 True로 바꿔 이 셀을 실행합니다.
STOP_DEMO = False
if STOP_DEMO and 'demo_process' in globals():
    demo_process.terminate()
    demo_log.close()
    print('이 노트북에서 시작한 임시 데모를 중지했습니다.')


## 12. 한계와 다음 실험

이번 결과는 좌표가 일반적으로 무효임을 뜻하지 않습니다. 모든 검증 지표가 0이어서 선택 기준이 회차를 구별하지 못했습니다. 단일 시드, 소량 합성 자료, 작은 모델, 한 가지 스타일과 제한된 모음이라는 한계가 있습니다. 의미 이해·문화 지식·사람의 가독성은 측정하지 않았습니다.

다음에는 작은 학습 부분집합을 실제로 암기할 수 있는지 확인하고, 음절 단위 분해부터 배치로 확장하는 학습이나 자료 확대를 검토할 수 있습니다. 이미 관찰한 테스트에 맞춰 설정을 고르고 같은 테스트를 새로운 미관찰 시험처럼 보고하면 안 됩니다. 후속 연구는 새 테스트를 먼저 분리해야 합니다.

기록에서 얻은 교훈은 **실행 성공, 손실 감소, 생성 성공을 구분해야 한다**는 것입니다. 개인적으로 느낀 점은 작성자가 실제 경험을 바탕으로 회고에 덧붙일 수 있습니다.

## 13. 참고문헌과 자료

- NAVER Cloud HyperCLOVA X Team (2025), [HyperCLOVA X THINK](https://arxiv.org/abs/2506.22403)
- Lee & Lee (2026), [Korean Jamo-Level Typographical Vulnerabilities](https://arxiv.org/abs/2608.30229v1)
- Hwang 외 (2025), [KRETA](https://aclanthology.org/2025.emnlp-main.1696/)
- Lu 외 (2025), [A Bounding Box is Worth One Token](https://aclanthology.org/2025.findings-acl.379/)

자모 취약성, 한국어 시각적 이해, 문자·위치 결합 연구를 배경으로 사용합니다. 이 연구들이 이번 자모 생성 방법의 성능을 직접 입증하지는 않습니다.